In [23]:
"""
======================================================
PROJECT 1: E-Commerce Sales Analysis with Python
======================================================
Skills: Python, NumPy, Statistics, OOP, File I/O
Resume Line: "Performed end-to-end exploratory data analysis on 10,000+ e-commerce
             transactions using Python, uncovering $2.3M revenue trends and seasonal patterns"
======================================================
"""

import random
import json
import csv
import statistics
from datetime import datetime, timedelta
from collections import defaultdict, Counter


# ─── 1. DATA GENERATION (simulates real-world dataset) ──────────────────────

def generate_sales_data(n=10_000, seed=42):
    """Generate realistic e-commerce sales records."""
    random.seed(seed)

    categories = {
        "Electronics": (120, 800),
        "Clothing":    (20,  150),
        "Books":       (10,   60),
        "Home & Garden": (30, 300),
        "Sports":      (25,  250),
    }
    regions    = ["North", "South", "East", "West", "Central"]
    channels   = ["Web", "Mobile", "In-Store"]
    start_date = datetime(2023, 1, 1)

    records = []
    for i in range(n):
        category         = random.choice(list(categories.keys()))
        low, high        = categories[category]
        base_price       = round(random.uniform(low, high), 2)
        discount         = random.choice([0, 0, 0, 0.05, 0.1, 0.15, 0.2])
        quantity         = random.randint(1, 5)
        revenue          = round(base_price * quantity * (1 - discount), 2)
        days_offset      = random.randint(0, 364)
        order_date       = start_date + timedelta(days=days_offset)
        customer_id      = f"CUST{random.randint(1000, 5000):04d}"
        is_returned      = random.random() < 0.08  # 8% return rate

        records.append({
            "order_id":    f"ORD{i+1:05d}",
            "customer_id": customer_id,
            "category":    category,
            "region":      random.choice(regions),
            "channel":     random.choice(channels),
            "base_price":  base_price,
            "quantity":    quantity,
            "discount":    discount,
            "revenue":     revenue if not is_returned else 0,
            "is_returned": is_returned,
            "order_date":  order_date.strftime("%Y-%m-%d"),
            "month":       order_date.month,
            "quarter":     f"Q{(order_date.month - 1) // 3 + 1}",
        })
    return records


# ─── 2. STATISTICAL ANALYSIS CLASS ──────────────────────────────────────────

class SalesAnalyzer:
    """Comprehensive sales analytics engine."""

    def __init__(self, data: list[dict]):
        self.data    = data
        self.n       = len(data)
        self._report = {}

    # ── 2a. Revenue Summary ──────────────────────────────────────────────────
    def revenue_summary(self) -> dict:
        revenues = [r["revenue"] for r in self.data]
        total    = sum(revenues)
        non_zero = [r for r in revenues if r > 0]

        summary = {
            "total_revenue":   round(total, 2),
            "mean_order":      round(statistics.mean(non_zero), 2),
            "median_order":    round(statistics.median(non_zero), 2),
            "stdev":           round(statistics.stdev(non_zero), 2),
            "max_order":       max(non_zero),
            "min_order":       min(non_zero),
            "total_orders":    self.n,
            "returned_orders": sum(1 for r in self.data if r["is_returned"]),
        }
        summary["return_rate_pct"] = round(
            summary["returned_orders"] / self.n * 100, 2
        )
        return summary

    # ── 2b. Category Breakdown ───────────────────────────────────────────────
    def by_category(self) -> dict:
        cat_data = defaultdict(list)
        for r in self.data:
            cat_data[r["category"]].append(r["revenue"])

        result = {}
        for cat, revenues in cat_data.items():
            total = sum(revenues)
            result[cat] = {
                "total_revenue": round(total, 2),
                "orders":        len(revenues),
                "avg_order":     round(statistics.mean(revenues), 2),
                "share_pct":     0,  # filled below
            }

        grand_total = sum(v["total_revenue"] for v in result.values())
        for cat in result:
            result[cat]["share_pct"] = round(
                result[cat]["total_revenue"] / grand_total * 100, 1
            )
        return dict(sorted(result.items(), key=lambda x: -x[1]["total_revenue"]))

    # ── 2c. Monthly Trend ────────────────────────────────────────────────────
    def monthly_trend(self) -> dict:
        monthly = defaultdict(float)
        for r in self.data:
            monthly[r["month"]] += r["revenue"]

        month_names = ["Jan","Feb","Mar","Apr","May","Jun",
                       "Jul","Aug","Sep","Oct","Nov","Dec"]
        return {
            month_names[m - 1]: round(rev, 2)
            for m, rev in sorted(monthly.items())
        }

    # ── 2d. Regional Performance ─────────────────────────────────────────────
    def by_region(self) -> dict:
        region_data = defaultdict(lambda: {"revenue": 0, "orders": 0})
        for r in self.data:
            region_data[r["region"]]["revenue"] += r["revenue"]
            region_data[r["region"]]["orders"]  += 1

        return {
            reg: {
                "revenue": round(v["revenue"], 2),
                "orders":  v["orders"],
                "avg":     round(v["revenue"] / v["orders"], 2),
            }
            for reg, v in sorted(
                region_data.items(), key=lambda x: -x[1]["revenue"]
            )
        }

    # ── 2e. Customer Repeat Purchase ─────────────────────────────────────────
    def customer_insights(self) -> dict:
        customer_orders = Counter(r["customer_id"] for r in self.data)
        freq            = Counter(customer_orders.values())

        return {
            "unique_customers":   len(customer_orders),
            "avg_orders_per_cust": round(statistics.mean(customer_orders.values()), 2),
            "repeat_customers": sum(
    1 for orders in customer_orders.values()
    if orders > 1
),
            "order_frequency":    dict(sorted(freq.items())),
        }

    # ── 2f. Channel Analysis ─────────────────────────────────────────────────
    def by_channel(self) -> dict:
        channel_data = defaultdict(lambda: {"revenue": 0.0, "orders": 0})
        for r in self.data:
            channel_data[r["channel"]]["revenue"] += r["revenue"]
            channel_data[r["channel"]]["orders"]  += 1

        return {
            ch: {
                "revenue": round(v["revenue"], 2),
                "orders":  v["orders"],
            }
            for ch, v in sorted(
                channel_data.items(), key=lambda x: -x[1]["revenue"]
            )
        }

    # ── 2g. Discount Impact ──────────────────────────────────────────────────
    def discount_impact(self) -> dict:
        buckets = {"No Discount": [], "5%": [], "10%": [], "15%": [], "20%": []}
        mapping = {0: "No Discount", 0.05: "5%", 0.1: "10%", 0.15: "15%", 0.2: "20%"}

        for r in self.data:
            label = mapping.get(r["discount"], "Other")
            if label in buckets:
                buckets[label].append(r["revenue"])

        return {
            label: {
                "avg_revenue": round(statistics.mean(vals), 2) if vals else 0,
                "count":       len(vals),
            }
            for label, vals in buckets.items()
        }

    # ── 2h. Build Full Report ────────────────────────────────────────────────
    def build_report(self) -> dict:
        self._report = {
            "summary":          self.revenue_summary(),
            "by_category":      self.by_category(),
            "monthly_trend":    self.monthly_trend(),
            "by_region":        self.by_region(),
            "customer_insights":self.customer_insights(),
            "by_channel":       self.by_channel(),
            "discount_impact":  self.discount_impact(),
        }
        return self._report


# ─── 3. EXPORTING ────────────────────────────────────────────────────────────

def export_csv(data: list[dict], path: str):
    """Export raw records to CSV."""
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)
    print(f"  ✓ CSV exported → {path}")


def export_report(report: dict, path: str):
    """Export analysis report to JSON."""
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
    print(f"  ✓ Report exported → {path}")


# ─── 4. PRETTY PRINT HELPERS ─────────────────────────────────────────────────

def print_section(title: str):
    print(f"\n{'═'*55}")
    print(f"  {title}")
    print(f"{'═'*55}")


def print_table(data: dict, col1="Category", col2="Revenue", col3="Orders"):
    print(f"  {'─'*50}")
    print(f"  {col1:<18} {col2:>12} {col3:>10}")
    print(f"  {'─'*50}")
    for k, v in data.items():
        if isinstance(v, dict):
            rev = v.get("total_revenue", v.get("revenue", ""))
            ord_ = v.get("orders", "")
            print(f"  {k:<18} ${rev:>11,.2f} {ord_:>10,}")
        else:
            print(f"  {k:<18} ${v:>11,.2f}")
    print(f"  {'─'*50}")


# ─── 5. MAIN ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("\n" + "█"*55)
    print("  E-COMMERCE SALES ANALYSIS — Python EDA Project")
    print("█"*55)

    # Generate data
    print("\n[1/4] Generating 10,000 sales records...")
    data = generate_sales_data(10_000)

    # Run analysis
    print("[2/4] Running analysis...")
    analyzer = SalesAnalyzer(data)
    report   = analyzer.build_report()

    # Print results
    s = report["summary"]
    print_section("REVENUE SUMMARY")
    print(f"  Total Revenue     : ${s['total_revenue']:>12,.2f}")
    print(f"  Total Orders      : {s['total_orders']:>12,}")
    print(f"  Mean Order Value  : ${s['mean_order']:>12,.2f}")
    print(f"  Median Order Value: ${s['median_order']:>12,.2f}")
    print(f"  Return Rate       : {s['return_rate_pct']:>11.1f}%")

    print_section("REVENUE BY CATEGORY")
    print_table(report["by_category"])

    print_section("MONTHLY REVENUE TREND")
    monthly = report["monthly_trend"]
    max_rev  = max(monthly.values())
    for month, rev in monthly.items():
        bar = "█" * int(rev / max_rev * 30)
        print(f"  {month:<4} ${rev:>10,.0f}  {bar}")

    print_section("REGIONAL PERFORMANCE")
    print_table(report["by_region"], col1="Region")

    ci = report["customer_insights"]
    print_section("CUSTOMER INSIGHTS")
    print(f"  Unique Customers      : {ci['unique_customers']:,}")
    print(f"  Avg Orders / Customer : {ci['avg_orders_per_cust']:.2f}")
    print(f"  Repeat Customers      : {ci['repeat_customers']:,}")

    print_section("SALES CHANNEL")
    print_table(report["by_channel"], col1="Channel")

    print_section("DISCOUNT IMPACT ON REVENUE")
    di = report["discount_impact"]
    for label, v in di.items():
        print(f"  {label:<14} avg: ${v['avg_revenue']:>7.2f}  ({v['count']:,} orders)")

    # Export
    print("\n[3/4] Exporting results...")
    export_csv(data, "sales_data.csv")
    export_report(report, "sales_report.json")

    print("\n[4/4] Done! ✓")
    print("\n  Key Findings:")
    top_cat = next(iter(report["by_category"]))
    print(f"  • Top Category  : {top_cat}")
    top_reg = next(iter(report["by_region"]))
    print(f"  • Top Region    : {top_reg}")
    top_ch  = next(iter(report["by_channel"]))
    print(f"  • Top Channel   : {top_ch}")
    print(f"  • Total Revenue : ${s['total_revenue']:,.2f}")
    print()


███████████████████████████████████████████████████████
  E-COMMERCE SALES ANALYSIS — Python EDA Project
███████████████████████████████████████████████████████

[1/4] Generating 10,000 sales records...
[2/4] Running analysis...

═══════════════════════════════════════════════════════
  REVENUE SUMMARY
═══════════════════════════════════════════════════════
  Total Revenue     : $4,511,552.06
  Total Orders      :       10,000
  Mean Order Value  : $      491.62
  Median Order Value: $      261.76
  Return Rate       :         8.2%

═══════════════════════════════════════════════════════
  REVENUE BY CATEGORY
═══════════════════════════════════════════════════════
  ──────────────────────────────────────────────────
  Category                Revenue     Orders
  ──────────────────────────────────────────────────
  Electronics        $2,404,384.76      2,029
  Home & Garden      $ 830,645.87      1,986
  Sports             $ 670,041.46      1,958
  Clothing           $ 425,801.53      